In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

models = {
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(), random_state=42),
    'LightGBM': LGBMClassifier(class_weight='balanced', random_state=42)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:,1]
    print(name, "ROC-AUC:", roc_auc_score(y_test, p))

RandomForest ROC-AUC: 0.6989093035107198
XGBoost ROC-AUC: 0.7153189032837806
[LightGBM] [Info] Number of positive: 209324, number of negative: 833561
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.148481 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2003
[LightGBM] [Info] Number of data points in the train set: 1042885, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM ROC-AUC: 0.7129802536393341


In [3]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

# use a 200k-row sample just for tuning, not the full dataset
sample_idx = X_train.sample(n=200000, random_state=42).index
X_sample = X_train.loc[sample_idx]
y_sample = y_train[X_train.index.get_indexer(sample_idx)]

param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1]
}

search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=(y_sample==0).sum()/(y_sample==1).sum(), random_state=42),
    param_grid,
    n_iter=6,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=2   # limit parallelism to reduce memory pressure
)

search.fit(X_sample, y_sample)
print("Best params:", search.best_params_)
print("Best CV ROC-AUC:", search.best_score_)

Best params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1}
Best CV ROC-AUC: 0.7109449344835922


In [4]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

best_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=42
)

best_model.fit(X_train, y_train)

probs = best_model.predict_proba(X_test)[:,1]
preds = best_model.predict(X_test)

print(classification_report(y_test, preds))
print("Final ROC-AUC:", roc_auc_score(y_test, probs))


              precision    recall  f1-score   support

           0       0.88      0.63      0.74    208391
           1       0.32      0.67      0.43     52331

    accuracy                           0.64    260722
   macro avg       0.60      0.65      0.58    260722
weighted avg       0.77      0.64      0.68    260722

Final ROC-AUC: 0.711911292177539


In [6]:
## Final Model: Tuned XGBoost
# - Params: n_estimators=100, max_depth=5, learning_rate=0.1
# - ROC-AUC: 0.712
# - Recall (default class): 0.67
# - Precision (default class): 0.32

In [7]:
import joblib

joblib.dump(best_model, '../models/credit_risk_model.pkl')
joblib.dump(list(X_train.columns), '../models/feature_columns.pkl')

['../models/feature_columns.pkl']